# Week 2 · Cleaning Jonas's oven log — the standard script

*Backhaus Kaya, Gummersbach · Eigene Darstellung, R. Bartnik 2026*

### What does "cleaning data" mean?
Think of a messy kitchen. Before you cook, you tidy up: you throw away empty packets, you put the salt back into the salt jar, and you read the labels again. Only then do you follow the recipe.

Data works the same way. A real file has extra spaces, words where numbers should be, rows typed twice and empty lines. **Cleaning data** means making the table tidy so that the computer counts correctly. The recipe comes afterwards. Today the recipe is: **rolls per hour**.

### The story
Head baker Jonas Weber writes every load of the deck oven into a spreadsheet. The roll window is 02:00–06:00 (4 hours per night). Aylin Kaya wants to know last week's numbers:

- **Actual output** = total rolls ÷ total hours of the roll windows → "x rolls per hour"
- **Utilization** = actual output ÷ design capacity (720 rolls per hour)
- **Efficiency** = actual output ÷ effective capacity (600 rolls per hour)

### How to load a file in Colab
**Easiest:** do nothing. If you have not uploaded a file, the notebook reads `FILE_NAME` directly from the course repository on the web.

Other ways:
1. **Upload:** click the folder icon 📁 on the left → click the upload icon → choose the file (for example `oven_log_A.csv`). Write its name into `FILE_NAME` below.
2. **Link:** paste a web link to the CSV file into `DATA_URL` below. The link must start with `http`.

### How to use this notebook
Change **only** the PARAMETERS cell. Then click **Runtime → Run all**. Read the printed messages from top to bottom.

| File | What to expect |
|---|---|
| `oven_log_A.csv` | Messy. This script cleans it. |
| `oven_log_B.csv` | A German export. 💡 **Hint:** set `SEP = ";"` and `DECIMAL = ","` in the PARAMETERS cell. |
| `oven_log_C.csv`, `oven_log_D.csv` | This script cannot clean them. It will tell you why. Then use the LLM prompt `05_LLM_prompt_diagnose_and_clean.md`. |


In [ ]:
# ======================= PARAMETERS — change only this cell =======================
DATA_URL  = "..."              # Way 2: paste a link to the CSV here (must start with "http"); keep "..." if you uploaded the file
FILE_NAME = "oven_log_A.csv"   # Way 1: the name of the file you uploaded with the folder icon
SEP       = ","                # the sign between columns: "," in most files, ";" in a German export (file B)
DECIMAL   = "."                # the decimal sign: "." in English files, "," in a German export (file B)
REPO_DATA = "https://raw.githubusercontent.com/TH-Koln-Bartnik/sscm-2026-student/main/week02/data/"  # the course files on the web

WINDOW_HOURS       = 4         # the roll window is 02:00-06:00, so 4 hours per night
DESIGN_CAPACITY    = 720       # rolls per hour the oven could make in ideal conditions (it is the bottleneck)
EFFECTIVE_CAPACITY = 600       # rolls per hour with planned losses (pre-heating, deck changes, cleaning)
MAX_ROLLS_PER_LOAD = 240       # 4 decks x 2 trays x 30 rolls = the most one load can hold
# ===================================================================================

## Step 1 · Load the file
We read the CSV file into a table. In pandas a table is called a **DataFrame**. We keep the original table as `raw` and never change it, so we can compare before and after.

In [ ]:
import os                          # "os" lets Python check whether a file exists
import pandas as pd                # pandas is Python's table tool; "pd" is its short name

STOP = False                       # STOP becomes True when a later step cannot run
PROBLEMS = []                      # a list where we collect every problem this script cannot fix
SETTINGS_HINT = False              # becomes True when the problem is only a wrong setting (SEP or DECIMAL)

if DATA_URL.startswith("http"):                          # did you paste a web link?
    source = DATA_URL                                    # then we read the file from the web
elif os.path.exists(FILE_NAME):                          # is the file in the main folder (in Colab: /content)?
    source = FILE_NAME                                   # then we read it from there
elif os.path.exists(os.path.join("data", FILE_NAME)):    # or is it in a sub-folder called "data"?
    source = os.path.join("data", FILE_NAME)             # then we read it from there
else:                                                    # no upload and no link
    source = REPO_DATA + FILE_NAME                       # then we read the file from the course repository on the web

if source is None:                                       # no file found
    STOP = True                                          # all later steps will be skipped
    print(f"❌ File '{FILE_NAME}' not found. Upload it with the folder icon, or paste a link into DATA_URL.")
else:                                                    # we have a file
    try:                                                 # "try" = attempt the next lines; if they fail, go to "except"
        raw = pd.read_csv(source, sep=SEP, decimal=DECIMAL)  # read the CSV file into a table called "raw"
        print(f"✅ Loaded '{source}': {raw.shape[0]} rows and {raw.shape[1]} columns.")  # say what we got
        display(raw.head(12))                            # show the first 12 rows so you can look at them
    except Exception as error:                           # reading failed (often: the wrong SEP sign)
        STOP = True                                      # all later steps will be skipped
        SETTINGS_HINT = True                             # this is probably a settings problem
        PROBLEMS.append("file could not be read")        # note the problem
        print("❌ pandas could not read the file:", str(error)[:120])  # show the start of the error message
        print('💡 Is this a German export? Then set SEP = ";" and DECIMAL = "," in the PARAMETERS cell.')

## Step 2 · Quick diagnosis — look before you clean
Like a doctor, we first look and count. We do not change anything yet.
- **shape** = number of rows and columns
- **dtypes** = the type of each column: `object` or `str` means text, `int64` and `float64` mean numbers. A number column that shows `object` or `str` contains some text.
- **missing values** = empty cells
- **duplicates** = rows that are an exact copy of another row

In [ ]:
if STOP:                                                    # an earlier step failed ...
    print("⏭️ Skipped: no data to look at.")                # ... so we skip this step
else:                                                       # otherwise we do the diagnosis
    print("Shape (rows, columns):", raw.shape)              # how big is the table?
    print("\nColumn types ('object' or 'str' = text, 'int64'/'float64' = numbers):")  # a heading
    print(raw.dtypes.to_string())                           # the type of every column
    print("\nMissing values (empty cells) per column:")     # a heading
    print(raw.isna().sum().to_string())                     # count the empty cells in every column
    blank = raw.isna().all(axis=1)                          # True for rows where EVERY cell is empty
    print(f"\nCompletely blank rows: {blank.sum()}")        # how many blank rows?
    copies = raw[~blank].duplicated().sum()                 # count rows that copy an earlier row (blank rows left out)
    print(f"Rows that are exact copies of an earlier row: {copies}")  # how many duplicates?

## Step 3 · Standardise the column names
`" ROLLS"`, `"Rolls"` and `"rolls"` look the same to us, but not to the computer. We make every name small letters, without spaces at the ends, with `_` instead of spaces inside. Then we check that all expected columns are there.

In [ ]:
EXPECTED = ["date", "load_no", "start_time", "end_time",      # the 9 columns of a correct oven log ...
            "deck_count_used", "trays", "rolls", "product", "operator"]  # ... one row per oven load

if STOP:                                                         # an earlier step failed ...
    print("⏭️ Skipped.")                                         # ... so we skip this step
else:                                                            # otherwise we tidy the names
    df = raw.copy()                                              # work on a copy called "df"; "raw" stays as it was
    df.columns = (df.columns.str.strip()                         # remove spaces before and after each name
                            .str.lower()                         # make all letters small
                            .str.replace(" ", "_"))              # replace spaces inside a name by "_"
    print("Column names now:", list(df.columns))                 # show the new names
    missing = [c for c in EXPECTED if c not in df.columns]       # which expected names are not there?
    if len(df.columns) == 1:                                     # only ONE column means the SEP sign is wrong
        STOP = True                                              # later steps cannot run
        SETTINGS_HINT = True                                     # it is a settings problem
        PROBLEMS.append("only one column (wrong SEP)")           # note the problem
        print('❌ Only 1 column. 💡 Set SEP = ";" and DECIMAL = "," (German export) and run again.')
    elif missing:                                                # some expected columns are missing
        STOP = True                                              # later steps cannot run
        PROBLEMS.append(f"wrong shape: missing columns {missing}")  # note the problem
        print(f"❌ These columns are missing: {missing}")       # say which ones
        load_columns = [c for c in df.columns if c.startswith("load_")]  # columns named load_01, load_02, ...
        if len(load_columns) > 2:                                # many "load_" columns = one column per load
            print(f"   The file has {len(df)} rows and {len(load_columns)} 'load_' columns:")  # describe it
            print("   it is in WIDE format (one row per night, one column per load).")       # name the problem
            print("   We need LONG format (one row per load). This script cannot reshape it.") # what is needed
        print("⚠️ The standard script cannot fix this file — try the LLM prompt.")  # where to go next
    else:                                                        # all 9 columns are there
        print("✅ All 9 expected columns found.")                # good news
        before = df.copy()                                       # keep a copy for the before/after report

## Step 4 · Strip text and units from numbers
Cells like `" 240 "` (spaces) or `"240 rolls"` (a unit) are text, not numbers. The computer cannot add them. We keep only the number part. In a German file `"8,0"` becomes `8.0`. We also tidy the text columns.

In [ ]:
NUMBER_COLUMNS = ["load_no", "deck_count_used", "trays", "rolls"]   # columns that must hold numbers
TEXT_COLUMNS = ["product", "operator"]                              # columns that hold words

if STOP:                                                            # an earlier step failed ...
    print("⏭️ Skipped.")                                            # ... so we skip this step
else:                                                               # otherwise we clean the numbers
    text_cells_fixed = 0                                            # a counter for cells we repair
    for col in NUMBER_COLUMNS:                                      # repeat the next lines for each number column
        text = df[col].astype(str).str.strip()                      # turn every cell into text and cut outer spaces
        if DECIMAL != ".":                                          # in a German file ...
            text = text.str.replace(DECIMAL, ".", regex=False)      # ... "8,0" becomes "8.0"
        plain = df[col].astype(str).str.fullmatch(r"-?\d+(\.\d+)?")  # True where the cell is a plain number (no spaces, no units)
        number = pd.to_numeric(text.str.extract(r"(-?\d+(?:\.\d+)?)")[0], errors="coerce")  # keep only the number part: "240 rolls" -> 240
        repaired = (~plain) & number.notna()                        # cells that had spaces or units but hold a number
        text_cells_fixed += int(repaired.sum())                     # add them to the counter
        if (number.dropna() % 1 == 0).all():                        # if all numbers are whole numbers (8.0, 240.0) ...
            number = number.astype("Int64")                         # ... store them as whole numbers (8, 240)
        df[col] = number                                            # put the clean numbers back into the table
    for col in TEXT_COLUMNS:                                        # repeat for each text column
        df[col] = df[col].astype("string").str.strip()              # cut spaces before and after the words
    df["product"] = df["product"].str.lower()                       # "Rolls " and "rolls" become the same word
    print(f"✅ Number cells repaired (spaces or units removed): {text_cells_fixed}")  # report the count
    print("Column types now:")                                      # a heading
    print(df[NUMBER_COLUMNS].dtypes.to_string())                    # the number columns should now be numbers

## Step 5 · Read dates and times
The computer must understand `2026-09-28` and `28.09.2026` as the same kind of thing: a date. We join the date with the start and end time of each load. Then we compute how long each load took. One load should take **20 minutes** (18 min baking + 2 min loading). A negative duration is impossible: the load would end before it starts.

In [ ]:
def make_timestamp(clock, day_text):                          # a small helper: date + time -> one timestamp
    clock = clock.astype(str).str.strip()                       # the time cells as text, without spaces
    text = clock.where(clock.str.len() > 5,                     # if a cell already holds a date and a time, keep it;
                       day_text + " " + clock)                  # if it holds only a time like "02:20", put the date in front
    return pd.to_datetime(text, format="%Y-%m-%d %H:%M", errors="coerce")  # read it; unreadable cells become NaT (empty)

if STOP:                                                        # an earlier step failed ...
    print("⏭️ Skipped.")                                        # ... so we skip this step
else:                                                           # otherwise we read the dates
    date_iso = pd.to_datetime(df["date"], format="%Y-%m-%d", errors="coerce")  # try the style 2026-09-28
    date_de = pd.to_datetime(df["date"], format="%d.%m.%Y", errors="coerce")   # try the German style 28.09.2026
    df["date"] = date_iso.fillna(date_de)                       # take whichever style worked
    day_text = df["date"].dt.strftime("%Y-%m-%d")               # the date written as text, for joining with the time
    df["start"] = make_timestamp(df["start_time"], day_text)    # date + start time of each load
    df["end"] = make_timestamp(df["end_time"], day_text)        # date + end time of each load
    df["duration_min"] = (df["end"] - df["start"]).dt.total_seconds() / 60  # minutes between start and end
    has_date = df["date"].notna()                               # rows that have a date (blank rows do not)
    unreadable = (has_date & df["start"].isna()).sum()          # rows with a date but no readable start time
    negative = (df["duration_min"] < 0).sum()                   # loads that "end before they start"
    too_long = (df["duration_min"] > 60).sum()                  # loads longer than one hour
    print("Durations in minutes (count of loads):")             # a heading
    print(df["duration_min"].value_counts().sort_index().to_string())  # how often each duration appears
    if unreadable:                                              # some times could not be read
        PROBLEMS.append(f"{unreadable} unreadable times")       # note the problem
        print(f"⚠️ {unreadable} rows have a time the script cannot read.")
    if negative:                                                # some loads end before they start
        PROBLEMS.append(f"{negative} negative durations")       # note the problem
        print(f"⚠️ {negative} loads have a NEGATIVE duration: the end time is before the start time.")
        print("   Look at the end_time column: something is wrong with the dates. This needs your judgement.")
    if too_long:                                                # some loads take more than an hour
        PROBLEMS.append(f"{too_long} loads longer than 60 min")  # note the problem
        print(f"⚠️ {too_long} loads take longer than 60 minutes.")
    if not (unreadable or negative or too_long):                # no time problem found
        print("✅ All dates and times read; no impossible durations.")

## Step 6 · Remove blank rows and duplicates
A blank row is an empty line in the spreadsheet. A duplicate is a load typed twice: if we keep it, we count its rolls twice. A missing operator name does not change the number of rolls, so we only mark it as "unknown".

In [ ]:
if STOP:                                                         # an earlier step failed ...
    print("⏭️ Skipped.")                                         # ... so we skip this step
else:                                                            # otherwise we remove the junk rows
    blank = df[["date", "load_no", "rolls"]].isna().all(axis=1)  # a row without date, load number and rolls is blank
    n_blank = int(blank.sum())                                   # how many blank rows?
    df = df[~blank]                                              # keep only the rows that are NOT blank
    n_dup = int(df.duplicated().sum())                           # how many rows copy an earlier row?
    df = df.drop_duplicates()                                    # keep only the first copy of each row
    n_no_operator = int(df["operator"].isna().sum())             # how many loads have no operator name?
    df["operator"] = df["operator"].fillna("unknown")            # write "unknown" into the empty name cells
    print(f"✅ Removed {n_blank} blank rows and {n_dup} duplicate rows; "
          f"marked {n_no_operator} missing operator(s) as 'unknown'.")  # report what we did

## Step 7 · Report: what changed?
Never trust a cleaning step you cannot see. This table compares the table before cleaning (after Step 3) with the table now. We also check that no load holds more rolls than the oven can hold (240).

In [ ]:
if STOP:                                                              # an earlier step failed ...
    print("⏭️ Skipped.")                                              # ... so we skip this step
else:                                                                 # otherwise we build the report
    rolls_before = pd.to_numeric(before["rolls"], errors="coerce")    # rolls before cleaning; text cells count as empty
    report = pd.DataFrame({                                           # a small table: one row per check
        "check": ["rows", "blank rows", "duplicate rows",             # the names of the checks ...
                  "number cells written as text", "missing operator (after: marked 'unknown')",
                  "rolls cells the computer can add", "total rolls"],
        "before": [len(before),                                       # rows before
                   int(before.isna().all(axis=1).sum()),              # blank rows before
                   n_dup,                                             # duplicates before (found in Step 6)
                   text_cells_fixed,                                  # text cells before (found in Step 4)
                   int(before["operator"].isna().sum()                # missing names before (blank rows left out)
                       - before.isna().all(axis=1).sum()),
                   int(rolls_before.notna().sum()),                   # rolls cells that were readable numbers
                   int(rolls_before.sum())],                          # the total the computer would have added
        "after": [len(df), 0, int(df.duplicated().sum()), 0,          # rows, blank rows, duplicates, text cells now
                  int((df["operator"] == "unknown").sum()),           # names marked "unknown"
                  int(df["rolls"].notna().sum()),                     # rolls cells that are numbers now
                  int(df["rolls"].sum())],                            # the clean total
    })
    display(report)                                                   # show the before/after table
    too_big = df[df["rolls"] > MAX_ROLLS_PER_LOAD]                    # loads with more rolls than the oven holds
    if len(too_big):                                                  # did we find any?
        PROBLEMS.append(f"{len(too_big)} loads above {MAX_ROLLS_PER_LOAD} rolls")  # note the problem
        print(f"⚠️ {len(too_big)} load(s) show more than {MAX_ROLLS_PER_LOAD} rolls — impossible for this oven:")
        display(too_big[EXPECTED])                                    # show those rows so you can judge them
    else:                                                             # nothing too big
        print(f"✅ No load is above {MAX_ROLLS_PER_LOAD} rolls.")      # good news
    print(f"Nights in the file: {df['date'].nunique()} · loads: {len(df)}")  # a last overview

## Step 8 · The recipe: rolls per hour, utilization, efficiency
Write every result as a sentence fraction: **output ÷ input = x [unit] per [unit]**. The ✅/❌ table compares your result with Aylin's reference week (540 rolls per hour, 75 %, 90 %).

In [ ]:
REFERENCE = {"actual output (rolls per hour)": 540.0,   # Aylin's reference week: actual output
             "utilization (%)": 75.0,                    # 540 / 720
             "efficiency (%)": 90.0}                     # 540 / 600

if STOP:                                                 # an earlier step failed ...
    print("⏭️ Skipped: no clean table, so no KPIs.")     # ... so we cannot compute anything
else:                                                    # otherwise we compute the KPIs
    total_rolls = df["rolls"].sum()                      # all rolls of the week
    nights = df["date"].nunique()                        # how many different nights are in the log
    hours = nights * WINDOW_HOURS                        # 4 hours of roll window per night
    actual = total_rolls / hours                         # actual output = rolls / hours
    utilization = actual / DESIGN_CAPACITY               # utilization = actual output / design capacity
    efficiency = actual / EFFECTIVE_CAPACITY             # efficiency = actual output / effective capacity
    print(f"Actual output: {total_rolls:,.0f} rolls ÷ {hours} hours ({nights} nights × {WINDOW_HOURS} h) "
          f"= {actual:,.1f} rolls per hour")             # sentence fraction 1
    print(f"Utilization:   {actual:,.1f} rolls per hour ÷ {DESIGN_CAPACITY} rolls per hour "
          f"= {utilization:.1%}")                        # sentence fraction 2
    print(f"Efficiency:    {actual:,.1f} rolls per hour ÷ {EFFECTIVE_CAPACITY} rolls per hour "
          f"= {efficiency:.1%}")                         # sentence fraction 3
    if efficiency > 1:                                   # more than 100 % of effective capacity?
        PROBLEMS.append("efficiency above 100 %")        # note the problem
        print("⚠️ Efficiency above 100 % is not plausible: look for a typo in the rolls column.")
    mine = [actual, 100 * utilization, 100 * efficiency]  # our three results as plain numbers
    check = pd.DataFrame({"KPI": list(REFERENCE),         # a table: KPI name ...
                          "your result": [round(v, 1) for v in mine],  # ... our result ...
                          "reference": list(REFERENCE.values())})      # ... and the reference
    check["match"] = ["✅" if abs(m - r) < 0.5 else "❌"                 # ✅ if we are within 0.5 of the reference
                      for m, r in zip(mine, REFERENCE.values())]
    display(check)                                       # show the check table

## Verdict

In [ ]:
if not PROBLEMS:                                                        # no problem collected?
    print("✅ The standard script cleaned this file. Compare your KPIs with the reference above.")
elif SETTINGS_HINT:                                                     # the problem is a setting
    print('💡 Change the PARAMETERS cell (SEP = ";" and DECIMAL = "," for a German export) and run all again.')
else:                                                                   # a problem the script cannot fix
    print("⚠️ The standard script cannot fix this file — try the LLM prompt.")  # where to go next
    print("   Problems found:")                                         # a heading
    for p in PROBLEMS:                                                  # repeat for each problem ...
        print("   -", p)                                                # ... and print it
    print("   Open 05_LLM_prompt_diagnose_and_clean.md, fill in the slots, and check the answer by hand.")

### Before you trust any result (also an LLM's result)
1. **Row count:** how many rows before, how many after? Can you explain every removed row?
2. **One row by hand:** pick one load in the original file and find it in the clean table. Same rolls, same times?
3. **Plausibility:** actual output must be below the design capacity of 720 rolls per hour, and efficiency is normally below 100 %.

*Claude (Anthropic) was used to draft and check this material; all content was reviewed by and is the responsibility of Roman Bartnik.*